<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/Python-2026/Practice/Statistica_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Вариант 3. Статистика (80 задач, 10 уровней)

> **Тема:** библиотека — книги, читатели, выдачи, штрафы. Один сквозной датасет от описательной статистики до A/B-аналитики и байесовских методов.
> **Цель:** покрыть **всю базовую теорию статистики**, необходимую аналитику / data scientist: от `mean/median` до `bootstrap`, `t-test`, `ANOVA`, `χ²`, регрессии и A/B-тестов.

> **Данные (одни для всех задач):** таблица `loans(loan_id, reader_id, book_id, genre, days_held, days_overdue, fine)`. Загружается из CSV через `pandas`. Три группы читателей (`A`, `B`, `C`) — подписчики на разные рассылки — используются в A/B-тестах.
> **Стек:** `numpy`, `scipy.stats`, `statsmodels`, `pandas`, `seaborn`, `matplotlib`.

---




## Уровень 0. Setup: генерация данных (0.1–0.5)

**Что изучается:** проектирование «статистически осмысленного» датасета, `numpy.random.default_rng`, распределения (`normal`, `lognormal`, `exponential`, `binomial`), контролируемые эффекты, пропуски/дубликаты, проверка через `describe`/`corr`, сохранение в CSV, `reset.py`.

**Задача 0.1 (спроектировать схему).** Прежде чем генерировать, нужно понять, **что** и **зачем**. Спроектируйте таблицу `loans` с полями: `loan_id` (int), `reader_id` (int), `book_id` (int), `genre` (категория: роман / детектив / фантастика / поэма), `hall` (категория: A / B / C — разные залы или группы рассылки), `days_held` (int, сколько дней книга на руках), `days_overdue` (int, ≥0, сколько дней просрочки), `fine` (float, штраф), `returned_on_time` (0/1), `group` (A / B / C, для A/B-тестов). Для каждого поля укажите: тип, диапазон, распределение, ожидаемую связь с другими полями. Ожидаемый результат: таблица «поле → тип → распределение → связь». В комментариях поясните: зачем `days_held` должен быть скошен (реальные сроки — log-normal); почему `days_overdue` — zero-inflated (большинство возвращают вовремя); почему `fine = days_overdue * rate + noise` (для регрессии); почему `group` влияет на `returned_on_time` (+2 пп для B). Подсказка: нарисуйте DAG: `group → returned_on_time ← days_held ← genre`.

**Задача 0.2 (сгенерировать `days_held` с выбросами).** Сгенерируйте `n = 1000` значений `days_held`: 95% — `normal(μ=14, σ=4)`, 5% — выбросы `uniform(30, 60)`. Используйте `np.random.default_rng(42)`. Ожидаемый вывод: `mean ≈ 15.2`, `median ≈ 13.8`, `mean > median` (правый хвост), `max = 58`. В комментариях поясните: почему `mean > median` при правом хвосте (задача 1); как выбросы ломают `std` (задача 2); почему `default_rng` лучше `np.random.seed` (современный API, изолированные генераторы); почему 5% выбросов — реалистично (реальные данные всегда «грязные»). Подсказка: `rng = np.random.default_rng(42); base = rng.normal(14, 4, 950); outliers = rng.uniform(30, 60, 50); days_held = np.concatenate([base, outliers])`.

**Задача 0.3 (сгенерировать `days_overdue` и `fine`).** Сгенерируйте `days_overdue`: 70% — `0`, 20% — `1–7` (`exponential(λ=1/3)`), 10% — `8–30` (`exponential(λ=1/10)`). `fine = days_overdue * 50 + normal(0, 10)` (только для `days_overdue > 0`, иначе `0`). Ожидаемый вывод: `P(days_overdue > 0) ≈ 0.30`; корреляция `days_overdue` и `fine` ≈ `0.95`; `fine[days_overdue=0] = 0`. В комментариях поясните: почему `days_overdue` — zero-inflated (большинство возвращают вовремя); почему `fine` — детерминированная функция + шум (реалистично, но есть связь для регрессии); почему `Pearson` между `days_overdue` и `fine` высокий, но не 1.0 (шум). Подсказка: `mask_zero = rng.random(n) < 0.7; days_overdue = np.where(mask_zero, 0, rng.exponential(3, n).astype(int))`.

**Задача 0.4 (сгенерировать `group` и `returned_on_time` с эффектом).** Три группы A/B/C по ~333 человека. Базовая вероятность возврата вовремя — 0.30. Группа B имеет **эффект +2 пп** (0.32), группа C — **без эффекта** (0.30). `returned_on_time ~ Binomial(1, p_group)`. Ожидаемый вывод: `p_A ≈ 0.30`, `p_B ≈ 0.32`, `p_C ≈ 0.30` (в пределах шума). В комментариях поясните: как задать эффект (сдвиг `p`); почему нужна мощность 0.8 (задача 36) — иначе эффект +2 пп не поймать при `n = 333`; почему `n = 333` на группу даст `p > 0.05` в большинстве прогонов (это **ожидаемо** — для задач A/B используйте `n = 6300` на группу, задача 66); почему `seed` важен (без него каждый прогон — новый результат). Подсказка: `p_group = {"A": 0.30, "B": 0.32, "C": 0.30}; returned = rng.binomial(1, [p_group[g] for g in groups])`.

**Задача 0.5 (добавить «грязь» + сохранить).** Добавьте: 2% пропусков в `days_held` (для `dropna`/`fillna`), 1% дубликатов строк (для `drop_duplicates`), 3 значения `fine = 99999` (выбросы для задач 4–5). Сохраните в `loans.csv` (`utf-8`, `index=False`). Проверьте: `df.shape == (1000, 10)`, `df.isna().sum()["days_held"] == 20`, `df.duplicated().sum() == 10`. Напишите `reset.py`: удаляет `loans.csv` и пересоздаёт заново (запуск дважды даёт тот же результат благодаря `seed`). Ожидаемый результат: CSV + `reset.py`, оба запускаются. В комментариях поясните: зачем «грязь» (реальные данные всегда грязные, навыки очистки — обязательны); почему `drop_duplicates` нужен перед `describe`; почему выбросы `99999` ломают `mean`, но не `median` (задача 8); почему `reset.py` с `seed` — основа воспроизводимости (тесты не должны «мигать»). Подсказка: `df.loc[rng.choice(df.index, 20, replace=False), "days_held"] = np.nan; df = pd.concat([df, df.sample(10)])`.

---

## Что конкретно должно быть в `loans.csv`

Готовый генератор (можешь запустить сразу):


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000

# Базовые поля
loan_id = np.arange(1, n + 1)
reader_id = rng.integers(1, 200, n)
book_id = rng.integers(1, 500, n)
genre = rng.choice(["роман", "детектив", "фантастика", "поэма"], n, p=[0.4, 0.3, 0.2, 0.1])
hall = rng.choice(["A", "B", "C"], n, p=[0.4, 0.35, 0.25])

# days_held: normal + выбросы
base = rng.normal(14, 4, 950)
outliers = rng.uniform(30, 60, 50)
days_held = np.clip(np.concatenate([base, outliers]), 1, None).round().astype(int)
rng.shuffle(days_held)

# days_overdue: zero-inflated
mask_zero = rng.random(n) < 0.7
days_overdue = np.where(mask_zero, 0, rng.exponential(3, n)).astype(int)

# fine: детерминированная функция + шум
fine = np.where(days_overdue > 0, days_overdue * 50 + rng.normal(0, 10, n), 0).round(2)

# A/B/C: эффект +2 пп у B
p_by_group = {"A": 0.30, "B": 0.32, "C": 0.30}
returned_on_time = np.array([rng.binomial(1, p_by_group[g]) for g in hall])

df = pd.DataFrame({
    "loan_id": loan_id, "reader_id": reader_id, "book_id": book_id,
    "genre": genre, "hall": hall, "group": hall,
    "days_held": days_held, "days_overdue": days_overdue,
    "fine": fine, "returned_on_time": returned_on_time,
})

# «Грязь»
df.loc[rng.choice(df.index, 20, replace=False), "days_held"] = np.nan
df = pd.concat([df, df.sample(10, random_state=42)], ignore_index=True)
df.loc[rng.choice(df.index, 3, replace=False), "fine"] = 99999

df.to_csv("loans.csv", index=False, encoding="utf-8")
print(df.shape)          # (1010, 10)
print(df.describe().T)   # sanity check
print(df["group"].value_counts())
print(df.groupby("group")["returned_on_time"].mean())  # A≈0.30, B≈0.32, C≈0.30


## 🟢 Уровень 1. Описательная статистика (1–8)

**Что изучается:** меры центральной тенденции (`mean`, `median`, `mode`), меры разброса (`variance`, `std`, `IQR`), квантили, выбросы, `describe()`, гистограммы и boxplot, устойчивость к выбросам.

**Задача 1 (среднее, медиана, мода).** Библиотеке нужно понять «типичный» срок выдачи книги. Загрузите `loans.csv`, возьмите столбец `days_held`. Вычислите `mean`, `median`, `mode` через `numpy`/`scipy.stats`. Ожидаемый вывод: `mean = 14.2`, `median = 12.0`, `mode = 10`. В комментариях поясните: почему `mean` ≠ `median` (распределение с «правым хвостом» — у нескольких должников огромные просрочки); почему `mode` — единственная мера, применимая к категориальным данным; когда `median` лучше `mean` (при выбросах). Подсказка: `from scipy import stats; stats.mode(days_held, keepdims=False)`.

**Задача 2 (дисперсия и стандартное отклонение).** Два зала библиотеки имеют одинаковый средний срок выдачи, но разный «разброс». Вычислите `variance` (по `ddof=0` и `ddof=1`), `std` для `days_held` в каждом зале. Ожидаемый вывод: `var_0 = 20.5`, `var_1 = 21.3`, `std = 4.53`. В комментариях поясните: чем `ddof=0` (population) отличается от `ddof=1` (sample) и почему в статистике почти всегда `ddof=1`; почему `std` в тех же единицах, что и данные, а `variance` — в квадратных; почему сравнивать разброс между разными единицами через `std` нельзя (нужен CV). Подсказка: `np.var(x, ddof=1)`, `np.std(x, ddof=1)`.

**Задача 3 (квантили и IQR).** Библиотека хочет знать «типичный» и «крайний» сроки. Вычислите 25-й, 50-й, 75-й перцентили (`np.percentile`), IQR = Q3 − Q1. Ожидаемый вывод: `Q1 = 7`, `Q3 = 18`, `IQR = 11`. В комментариях поясните: чем перцентиль отличается от квантиля (одно и то же, разные единицы); почему IQR устойчив к выбросам, а `std` — нет; как 1.5·IQR даёт границы «усов» boxplot. Подсказка: `q1, q3 = np.percentile(x, [25, 75])`.

**Задача 4 (выбросы: IQR-метод).** В библиотеке есть «злостные должники» с огромными просрочками, и они искажают среднее. Найдите выбросы по правилу 1.5·IQR. Ожидаемый вывод: `Найдено 12 выбросов`, `Нижняя граница = −9.5`, `Верхняя = 34.5`. В комментариях поясните: почему 1.5 — эмпирический коэффициент (Тьюки); почему нижняя граница может быть отрицательной (для `days_held` это нормально — просто нет таких значений); чем IQR-метод отличается от z-score (см. задачу 5). Подсказка: `lower = q1 - 1.5 * iqr; upper = q3 + 1.5 * iqr`.

**Задача 5 (выбросы: z-score).** Альтернативный метод — стандартизация. Вычислите z-score для `days_held` и найдите `|z| > 3`. Ожидаемый вывод: `Найдено 8 выбросов`, `z_max = 4.7`. В комментариях поясните: почему z-score предполагает нормальность (иначе работает плохо); почему `|z| > 3` — правило «трёх сигм»; чем z-score лучше IQR (учитывает форму), чем хуже (чувствителен к самим выбросам); что такое `robust z-score` (median + MAD). Подсказка: `z = (x - x.mean()) / x.std(ddof=1)`.

**Задача 6 (describe + гистограмма).** Быстрый обзор числового столбца. Сделайте `df["days_held"].describe()` и постройте гистограмму через `seaborn.histplot` с `kde=True`. Ожидаемый вывод: таблица `count/mean/std/min/25%/50%/75%/max` + график. В комментариях поясните: что показывает каждый квантиль в `describe`; зачем `kde=True` (сглаженная оценка плотности); почему `bins` важен (слишком много — шум, слишком мало — теряется форма); как по гистограмме понять, нормальное ли распределение (симметрия, колокол). Подсказка: `sns.histplot(df["days_held"], kde=True, bins=30)`.

**Задача 7 (boxplot: сравнение групп).** Сравните сроки выдачи по трём жанрам: `роман`, `детектив`, `фантастика`. Постройте `sns.boxplot(x="genre", y="days_held")`. Ожидаемый вывод: три «ящика» с усами и точками-выбросами. В комментариях поясните: что показывает каждый элемент boxplot (медиана, Q1, Q3, усы, выбросы); почему boxplot — «пятичисловая сводка»; когда boxplot лучше violinplot (компактнее) и когда хуже (скрывает бимодальность). Подсказка: `sns.boxplot(data=df, x="genre", y="days_held")`.

**Задача 8 (устойчивость к выбросам).** Покажите на практике: одно «аномальное» значение ломает `mean`, но не `median`. Добавьте к выборке `days_held` одно значение `1000` и сравните `mean` и `median` до/после. Ожидаемый вывод: `mean: 14.2 → 62.5`, `median: 12.0 → 12.0`. В комментариях поясните: почему `median` называется «устойчивой» (robust) статистикой; почему в отчётах по зарплатам/ценам используют медиану; почему для «среднего чека» иногда всё-таки нужен `mean` (важна общая сумма); что такое `trimmed mean` как компромисс. Подсказка: `from scipy.stats import trim_mean; trim_mean(x, 0.1)`.

---

## 🔵 Уровень 2. Распределения (9–16)

**Что изучается:** нормальное, биномиальное, Пуассона, равномерное, экспоненциальное, t-распределение, χ², QQ-plot, проверка нормальности (`Shapiro-Wilk`, `Kolmogorov-Smirnov`), `pdf/cdf/ppf/rvs`.

**Задача 9 (нормальное распределение: pdf/cdf/ppf).** Сроки выдачи близки к нормальному `N(μ=14, σ=4)`. Вычислите: плотность в точке `x=14`, вероятность `P(X ≤ 10)`, квантиль 95%. Ожидаемый вывод: `pdf(14) = 0.0997`, `cdf(10) = 0.1587`, `ppf(0.95) = 20.58`. В комментариях поясните: чем `pdf` отличается от `cdf`; что такое `ppf` (обратная `cdf`); почему `cdf(mean) = 0.5` для симметричного распределения; как правило 68-95-99.7 работает. Подсказка: `from scipy.stats import norm; norm.pdf(14, 14, 4)`, `norm.cdf(10, 14, 4)`, `norm.ppf(0.95, 14, 4)`.

**Задача 10 (стандартизация: Z).** Переведите `days_held` в Z-шкалу: `z = (x - μ) / σ`. Проверьте: `P(Z ≤ 1.96) ≈ 0.975`, `P(-1.96 ≤ Z ≤ 1.96) ≈ 0.95`. Ожидаемый вывод: `0.975`, `0.95`. В комментариях поясните: зачем стандартизация (сравнение разных шкал, t-test, регрессия); почему 1.96 — «магическое» число (95% интервал); как читать Z-таблицу вручную и зачем она, если есть `scipy.stats.norm.cdf`. Подсказка: `norm.cdf(1.96) - norm.cdf(-1.96)`.

**Задача 11 (биномиальное распределение).** 30% читателей возвращают книгу вовремя. Из 100 выдач — сколько «вовремя» ожидается? Вычислите `mean`, `std`, `P(X = 30)`, `P(X ≥ 40)`. Ожидаемый вывод: `mean = 30`, `std = 4.58`, `P(X = 30) = 0.072`, `P(X ≥ 40) = 0.021`. В комментариях поясните: параметры `n` и `p`; почему `mean = n·p`, `std = √(n·p·(1−p))`; когда биномиальное приближается нормальным (`n·p ≥ 5` и `n·(1−p) ≥ 5`); где это примemenимо (конверсия, доля возвратов, клики). Подсказка: `from scipy.stats import binom; binom.pmf(30, 100, 0.3)`, `1 - binom.cdf(39, 100, 0.3)`.

**Задача 12 (распределение Пуассона).** В библиотеку в среднем приходит 5 новых книг в день. Вычислите `P(X = 0)`, `P(X ≤ 3)`, `P(X > 10)`. Ожидаемый вывод: `0.0067`, `0.265`, `0.0137`. В комментариях поясните: параметр `λ` (интенсивность); почему `mean = variance = λ`; когда Пуассон применим (редкие события, независимые, фиксированный интервал); как связано с биномиальным при `n → ∞`, `p → 0`. Подсказка: `from scipy.stats import poisson; poisson.pmf(0, 5)`, `poisson.cdf(3, 5)`.

**Задача 13 (экспоненциальное распределение).** Время до следующего возврата книги — экспоненциальное с `λ = 1/14` (в среднем 14 дней). Вычислите `P(T ≤ 7)`, `P(T > 21)`, медиану. Ожидаемый вывод: `0.393`, `0.223`, `9.70`. В комментариях поясните: связь с Пуассоном (интервалы между событиями); почему `median = ln(2)/λ < mean = 1/λ` (право-скошенное); свойство «отсутствия памяти»; где применяется (время до отказа, время ожидания). Подсказка: `from scipy.stats import expon; expon.cdf(7, scale=14)`, `expon.ppf(0.5, scale=14)`.

**Задача 14 (t-распределение Стьюдента).** При малых выборках (`n < 30`) вместо нормального используют `t` с `df = n−1`. Сравните квантиль 95% для `N(0,1)` и `t` при `df = 5, 10, 30, ∞`. Ожидаемый вывод: `1.645` vs `2.015` (df=5), `1.812` (df=10), `1.697` (df=30). В комментариях поясните: почему `t` «толще» нормального на хвостах; почему при `df → ∞` `t → N(0,1)`; зачем `df = n−1` (потеря степени свободы на оценку среднего); где применяется (`t-test`, доверительные интервалы). Подсказка: `from scipy.stats import t; t.ppf(0.975, df=5)`.

**Задача 15 (χ²-распределение).** χ² используется в тестах на независимость и согласие. Постройте `χ²` с `df = 1, 5, 10` и найдите квантиль 95%. Ожидаемый вывод: `3.84`, `11.07`, `18.31`. В комментариях поясните: что такое «степени свободы»; почему `χ²` — распределение **суммы квадратов** нормальных величин; как `df` связан с числом категорий (для таблицы `r × c` — `(r−1)(c−1)`); где применяется (тесты независимости, согласия, дисперсии). Подсказка: `from scipy.stats import chi2; chi2.ppf(0.95, df=5)`.

**Задача 16 (QQ-plot + Shapiro-Wilk).** Проверьте, нормально ли распределены `days_held`. Постройте QQ-plot и выполните `shapiro`. Ожидаемый вывод: QQ-plot с точками около прямой; `p-value = 0.03` (отвергаем нормальность на α=0.05). В комментариях поясните: что показывает QQ-plot (сравнение квантилей выборки и теоретического распределения); как читать отклонения (S-образная кривая — тяжёлые хвосты, «ступенька» — бимодальность); что такое `p-value` в тесте на нормальность (**не** «вероятность нормальности»!); почему `Shapiro-Wilk` лучше `Kolmogorov-Smirnov` на малых выборках. Подсказка: `from scipy.stats import shapiro; shapiro(x)`; `import statsmodels.api as sm; sm.qqplot(x, line="45")`.

---

## 🟣 Уровень 3. Вероятность и ЦПТ (17–24)

**Что изучается:** вероятность, условная вероятность, формула Байеса, независимость, ЦПТ, закон больших чисел, стандартная ошибка, `se = σ/√n`, симуляции.

**Задача 17 (базовая вероятность).** Из 1000 книг 300 — «роман», 200 — «детектив». Какова вероятность взять «роман»? А «роман» или «детектив»? Ожидаемый вывод: `P(роман) = 0.3`, `P(роман ∪ детектив) = 0.5` (если события несовместны). В комментариях поясните: аксиомы Колмогорова (`P(Ω) = 1`, `P(A) ≥ 0`, `P(A ∪ B) = P(A) + P(B) − P(A ∩ B)`); когда `P(A ∪ B) = P(A) + P(B)` (несовместные); почему `P(¬A) = 1 − P(A)`. Подсказка: `p = 300 / 1000`.

**Задача 18 (условная вероятность).** 40% читателей — из Москвы, 25% — из Москвы и берут «роман». Какова `P(роман | Москва)`? Ожидаемый вывод: `0.625`. В комментариях поясните: формула `P(A|B) = P(A∩B) / P(B)`; почему `P(A|B) ≠ P(B|A)` (классическая ошибка); пример: `P(роман | Москва)` ≠ `P(Москва | роман)`; где это критично (медицина, спам-фильтры, A/B). Подсказка: `p = 0.25 / 0.40`.

**Задача 19 (формула Байеса).** Тест на «должника» имеет `sensitivity = 0.95`, `specificity = 0.90`. Должников в популяции 10%. Какова `P(должник | тест+)`? Ожидаемый вывод: `≈ 0.514` (а не 0.95!). В комментариях поясните: формулу Байеса `P(A|B) = P(B|A)·P(A) / P(B)`; почему при низкой базовой ставке даже хороший тест даёт много ложноположительных; что такое `base rate fallacy`; как это связано с COVID-тестами и скринингом. Подсказка: `p = (0.95 * 0.10) / (0.95 * 0.10 + 0.10 * 0.90)`.

**Задача 20 (независимость).** События A = «читатель из Москвы», B = «берёт роман». Проверьте независимость: `P(A∩B) = P(A)·P(B)`? Ожидаемый вывод: `P(A∩B) = 0.25`, `P(A)·P(B) = 0.12` → не независимы. В комментариях поясните: определение независимости; почему независимость — сильное утверждение (не «слабая связь», а **отсутствие** связи); как проверить независимость статистически (`χ²`-тест, задача 49); почему «независимость» ≠ «некоррелированность» (для нормальных — совпадает, в общем — нет). Подсказка: сравните `p_ab` и `p_a * p_b`.

**Задача 21 (ЦПТ: симуляция).** Центральная предельная теорема — фундамент почти всей статистики. Сгенерируйте 10 000 выборок размера `n = 30` из экспоненциального распределения, посчитайте средние, постройте гистограмму. Ожидаемый вывод: гистограмма средних — **колокол**, хотя исходное распределение — экспоненциальное. В комментариях поясните: формулировку ЦПТ (сумма/среднее `n` независимых одинаково распределённых величин → нормальное при `n → ∞`); почему это работает для **любого** исходного распределения с конечной дисперсией; почему `n = 30` — эмпирическое правило, но для сильно скошенных нужно больше; как ЦПТ обосновывает `t-test` и доверительные интервалы. Подсказка: `means = [np.random.exponential(scale=14, size=30).mean() for _ in range(10000)]`.

**Задача 22 (закон больших чисел).** С ростом `n` выборочное среднее приближается к теоретическому. Сгенерируйте выборки размеров `10, 100, 1000, 10000` из `N(14, 4)`, постройте средние. Ожидаемый вывод: с ростом `n` разброс средних сужается. В комментариях поясните: формулировку ЗБЧ (среднее сходится к `μ` по вероятности); чем ЗБЧ отличается от ЦПТ (ЗБЧ — про сходимость, ЦПТ — про **распределение** отклонения); почему казино всегда в плюсе (ЗБЧ для них); как это связано со `se = σ/√n`. Подсказка: `np.random.normal(14, 4, size=n).mean()`.

**Задача 23 (стандартная ошибка).** Стандартная ошибка среднего `se = σ/√n` показывает, насколько выборочное среднее отклоняется от истинного. Вычислите `se` для `n = 10, 100, 1000` при `σ = 4`. Ожидаемый вывод: `1.26`, `0.4`, `0.126`. В комментариях поясните: чем `se` отличается от `std` (`se` — про **среднее**, `std` — про **отдельное наблюдение**); почему `se` уменьшается как `√n` (чтобы уменьшить в 2 раза, нужно в 4 раза больше данных); как `se` используется в доверительных интервалах (задача 25); почему на практике `σ` неизвестна и заменяется на `s` (выборочное `std`). Подсказка: `se = sigma / np.sqrt(n)`.

**Задача 24 (bootstrap: идея).** Bootstrap — универсальный метод оценки распределения статистики без предположений о форме. Сгенерируйте 10 000 bootstrap-выборок из `days_held`, посчитайте `mean` в каждой, постройте гистограмму. Ожидаемый вывод: гистограмма средних, близкая к нормальной (по ЦПТ). В комментариях поясните: идею bootstrap (ресемплирование **с возвращением**); почему он работает (эмпирическое распределение ≈ истинное при большом `n`); когда bootstrap незаменим (медиана, IQR, корреляция — нет формул); почему `bootstrap` — «вычислительная» статистика. Подсказка: `boot_means = [np.random.choice(x, size=len(x), replace=True).mean() for _ in range(10000)]`.

---

## 🟠 Уровень 4. Выборка и доверительные интервалы (25–32)

**Что изучается:** популяция vs выборка, смещение выборки, доверительные интервалы для среднего, доли, разности средних, `bootstrap`-интервалы, интерпретация (частотная vs байесовская).

**Задача 25 (доверительный интервал для среднего: σ известна).** Если `σ` известна, `95% ДИ = x̄ ± 1.96·σ/√n`. Вычислите ДИ для `days_held` при `σ = 4`, `n = 100`, `x̄ = 14.2`. Ожидаемый вывод: `[13.42, 14.98]`. В комментариях поясните: **что значит** «95% ДИ» (при повторных выборках 95% интервалов накроют `μ` — **не** «вероятность 95%, что `μ` внутри»); почему для 99% нужно `2.58·σ/√n`; почему ширина интервала растёт с `σ` и падает с `n`. Подсказка: `ci = (x_bar - 1.96*se, x_bar + 1.96*se)`.

**Задача 26 (ДИ для среднего: σ неизвестна, t).** На практике `σ` неизвестна — используют `t` и выборочное `s`. Вычислите 95% ДИ через `scipy.stats.t.interval`. Ожидаемый вывод: `[13.31, 15.09]` (шире, чем в 25). В комментариях поясните: почему `t` шире нормального на малых `n`; почему `df = n − 1`; как `scipy.stats.t.interval` возвращает сразу интервал; почему при `n > 100` разница с нормальным почти исчезает. Подсказка: `from scipy.stats import t; t.interval(0.95, df=n-1, loc=x_bar, scale=s/np.sqrt(n))`.

**Задача 27 (ДИ для доли).** 30% из 100 читателей возвращают книги вовремя. Постройте 95% ДИ для доли (`Wald` и `Wilson`). Ожидаемый вывод: `Wald: [0.21, 0.39]`, `Wilson: [0.216, 0.399]`. В комментариях поясните: почему `Wald` плох для долей, близких к 0/1; почему `Wilson` — стандарт (лучше покрытие); почему нельзя использовать формулу для среднего напрямую (дисперсия доли зависит от `p`); как `statsmodels` делает это одной строкой. Подсказка: `from statsmodels.stats.proportion import proportion_confint; proportion_confint(30, 100, method="wilson")`.

**Задача 28 (ДИ для разности средних).** Сравните средний срок выдачи в двух залах. Вычислите 95% ДИ для `μ1 − μ2`. Ожидаемый вывод: `[−0.8, 3.2]` — интервал содержит 0 → нет значимой разницы. В комментариях поясните: почему интервал, содержащий 0, означает «разница не значима»; чем `pooled` vs `Welch` (`equal_var=True/False`) — какой использовать по умолчанию (`Welch`); как интервал связан с `t-test` (задача 41); почему интервал информативнее `p-value`. Подсказка: `from statsmodels.stats.weightstats import CompareMeans; CompareMeans.from_data(x1, x2).tconfint_diff(alpha=0.05)`.

**Задача 29 (bootstrap-ДИ).** Постройте 95% bootstrap-ДИ для медианы `days_held` (для неё нет аналитической формулы). Ожидаемый вывод: `[11.0, 13.0]`. В комментариях поясните: как работает percentile bootstrap (2.5-й и 97.5-й перцентили бутстрэп-распределения); чем `BCa` лучше (корректирует смещение); когда bootstrap незаменим (медиана, IQR, корреляция, кастомная статистика); почему нужно ≥ 1000 итераций (обычно 10 000). Подсказка: `boot_medians = [np.median(np.random.choice(x, size=len(x), replace=True)) for _ in range(10000)]; np.percentile(boot_medians, [2.5, 97.5])`.

**Задача 30 (смещение выборки).** Библиотека опрашивает читателей через онлайн-форму. Почему средняя оценка может быть смещена? Приведите пример `selection bias`, `response bias`, `survivorship bias`. Ожидаемый вывод: текстовое объяснение + численный пример (средняя оценка в онлайн-опросе 4.5, в бумажном — 3.8). В комментариях поясните: чем «выборка» отличается от «популяции»; почему случайная выборка — золотой стандарт; почему «удобная» выборка (первые 100 пришедших) плоха; как исправить (`stratified sampling`, взвешивание). Подсказка: `np.random.choice(population, size=n, replace=False)`.

**Задача 31 (размер выборки).** Сколько читателей нужно опросить, чтобы оценить средний срок выдачи с точностью ±1 день при 95% уверенности и `σ = 4`? Ожидаемый вывод: `n ≈ 62` (формула `n = (z·σ/E)²`). В комментариях поясните: вывод формулы; почему размер растёт как квадрат точности (уменьшить `E` в 2 раза → `n` в 4 раза); как учитывать конечную популяцию (`finite population correction`); как это связано с A/B (задача 65). Подсказка: `n = (1.96 * 4 / 1) ** 2`.

**Задача 32 (частотный vs байесовский ДИ).** Покажите разницу интерпретаций: «95% частотный ДИ» vs «95% байесовский credible interval». Ожидаемый вывод: текстовое сравнение + пример с байесовским интервалом через `PyMC` (обзорно). В комментариях поясните: частотный — «95% таких интервалов накроют μ»; байесовский — «вероятность 95%, что μ в интервале» (при данном prior); почему байесовский интервал требует prior; когда байесовский подход полезен (малые выборки, A/B с историей); почему нельзя интерпретировать частотный ДИ как «вероятность». Подсказка: `import pymc as pm` — обзорно; главное — понять концептуальную разницу.

---

## 🔴 Уровень 5. Гипотезы: база (33–40)

**Что изучается:** H₀/H₁, p-value, α, ошибки I/II рода, мощность, one-tailed vs two-tailed, `p-value` ≠ «вероятность H₀», критические значения, effect size.

**Задача 33 (H₀ и H₁).** Библиотека хочет проверить: средний срок выдачи **изменился** после новой политики. Сформулируйте H₀ и H₁. Ожидаемый вывод: `H₀: μ = 14`, `H₁: μ ≠ 14` (two-tailed). В комментариях поясните: почему H₀ — всегда «нет эффекта» / «равенство»; почему нельзя «доказать H₀» (только «не отвергнуть»); чем one-tailed (`μ > 14` или `μ < 14`) отличается от two-tailed; когда one-tailed уместен (только если **до** сбора данных была гипотеза о направлении). Подсказка: H₀ формулируется как «равенство», H₁ — как «отличие».

**Задача 34 (p-value: что это).** На выборке `n = 100` получено `x̄ = 14.8`, `s = 4`, `H₀: μ = 14`. Вычислите `p-value` (two-tailed). Ожидаемый вывод: `p ≈ 0.046`. В комментариях поясните: **что значит** p-value («вероятность получить такое или более экстремальное значение при условии H₀»); почему **не** «вероятность H₀»; почему **не** «вероятность ошибки»; почему `p = 0.05` — конвенция, а не магия; почему «p < 0.05» ≠ «эффект большой» (нужен effect size, задача 40). Подсказка: `from scipy.stats import ttest_1samp; ttest_1samp(x, 14)`.

**Задача 35 (α, ошибки I/II рода).** При `α = 0.05` постройте таблицу: что происходит при H₀ верна/неверна и решении отвергнуть/не отвергнуть. Ожидаемый вывод: 2×2 таблица + определения: ошибка I рода = отвергли верную H₀ (α), ошибка II рода = не отвергли неверную H₀ (β). В комментариях поясните: почему α — «уровень значимости» (мы **выбираем**); почему β зависит от эффекта и `n`; почему `1 − β` — мощность (задача 36); почему уменьшение α увеличивает β (компромисс); как связаны с медицинскими тестами (ложноположительный vs ложноотрицательный). Подсказка: нарисуйте 2×2 таблицу.

**Задача 36 (мощность теста).** Мощность `= 1 − β` — вероятность отвергнуть H₀, когда она действительно неверна. Вычислите мощность `t-test` для `μ₁ − μ₀ = 0.5`, `σ = 4`, `n = 100`, `α = 0.05`. Ожидаемый вывод: `power ≈ 0.13` (низкая!). В комментариях поясните: почему `n = 100` недостаточно для маленького эффекта; как `n`, `α`, effect size и `σ` влияют на мощность; почему «незначимый результат» ≠ «нет эффекта» (может, мощности не хватило); как использовать `statsmodels` для power analysis до эксперимента. Подсказка: `from statsmodels.stats.power import TTestIndPower; TTestIndPower().power(effect_size=0.125, nobs1=100, alpha=0.05, ratio=1)`.

**Задача 37 (one-tailed vs two-tailed).** Пересчитайте задачу 34 для one-tailed (`H₁: μ > 14`). Ожидаемый вывод: `p ≈ 0.023` (в 2 раза меньше). В комментариях поясните: почему one-tailed даёт меньший `p-value`; почему нельзя «переключиться» на one-tailed **после** просмотра данных (p-hacking); когда one-tailed оправдан (только при сильной предварительной гипотезе); почему регуляторы и журналы требуют two-tailed по умолчанию. Подсказка: `ttest_1samp(x, 14, alternative="greater")`.

**Задача 38 (критические значения).** Для `α = 0.05` (two-tailed) и `df = 99` найдите критическое значение `t`. Ожидаемый вывод: `t_crit ≈ 1.984`. В комментариях поясните: как критическое значение связано с `p-value` (`|t| > t_crit` ⇔ `p < α`); почему для больших `n` `t_crit → 1.96`; как читать t-таблицу вручную (строка — `df`, столбец — `α`); почему в коде почти всегда `p-value`, а не критическое значение. Подсказка: `from scipy.stats import t; t.ppf(0.975, df=99)`.

**Задача 39 (множественные сравнения: проблема).** Если проверить 20 гипотез при `α = 0.05`, ожидается 1 ложноположительный результат случайно. Продемонстрируйте симуляцией: 20 тестов на случайных данных, посчитайте, сколько «значимых». Ожидаемый вывод: `~1 из 20` (ложноположительный). В комментариях поясните: что такое `family-wise error rate` (FWER); почему `α` «инфлируется» при множественных сравнениях; почему `p-hacking` — серьёзная проблема (публикационная предвзятость); как исправить (`Bonferroni`, `Holm`, `FDR` — задача 70). Подсказка: `sum(ttest_1samp(np.random.normal(0, 1, 100), 0).pvalue < 0.05 for _ in range(20))`.

**Задача 40 (effect size: Cohen's d).** `p-value` показывает «есть ли эффект», но не «насколько большой». Вычислите Cohen's `d` для разности средних `x̄₁ − x̄₂ = 0.5`, `s_pooled = 4`. Ожидаемый вывод: `d = 0.125` (маленький эффект). В комментариях поясните: формулу `d = (x̄₁ − x̄₂) / s_pooled`; правила Cohen (0.2 — малый, 0.5 — средний, 0.8 — большой); почему «значимый, но маленький» эффект может быть бесполезен на практике; другие меры (`η²`, `r`, `odds ratio`); почему `effect size` обязателен в отчётах. Подсказка: `d = (x1.mean() - x2.mean()) / np.sqrt(((len(x1)-1)*x1.var(ddof=1) + (len(x2)-1)*x2.var(ddof=1)) / (len(x1)+len(x2)-2))`.

---

## 🟣 Уровень 6. Параметрические тесты (41–48)

**Что изучается:** 1-sample t-test, 2-sample t-test (Student и Welch), paired t-test, one-way ANOVA, проверка предпосылок, `scipy.stats.ttest_ind`, `f_oneway`, `statsmodels`.

**Задача 41 (1-sample t-test).** Проверьте гипотезу: средний срок выдачи `μ = 14` (H₀) против `μ ≠ 14`. Выполните `ttest_1samp`, выведите `t`, `p-value`, 95% ДИ. Ожидаемый вывод: `t = 1.99`, `p = 0.049`, `CI = [14.01, 15.59]`. В комментариях поясните: формулу `t = (x̄ − μ₀) / (s/√n)`; когда `t-test` применим (нормальность или `n > 30` по ЦПТ); почему при `p < 0.05` «отвергаем H₀», а не «доказываем H₁»; как отчёт выглядит в статье (`t(99) = 1.99, p = 0.049`). Подсказка: `ttest_1samp(x, 14)`.

**Задача 42 (2-sample t-test: Student).** Сравните средний срок выдачи в двух залах (независимые выборки, предполагаем равные дисперсии). Выполните `ttest_ind(equal_var=True)`. Ожидаемый вывод: `t = 0.42`, `p = 0.67` (нет разницы). В комментариях поясните: почему `Student` предполагает равные дисперсии; почему на практике почти всегда используют `Welch` (задача 43); что такое `pooled variance`; как интерпретировать `t`: «на сколько `se` разность средних отличается от 0». Подсказка: `ttest_ind(x1, x2, equal_var=True)`.

**Задача 43 (2-sample t-test: Welch).** Повторите 42 с `equal_var=False` (Welch). Ожидаемый вывод: `t = 0.42`, `p = 0.67` (почти то же). В комментариях поясните: чем Welch лучше (`df` не целое, не предполагает равных дисперсий); почему Welch — **дефолт** в `scipy` (`equal_var=True` — историческая ошибка); когда разница существенна (сильно разные `σ` и `n`); почему в отчётах указывают `Welch's t-test`. Подсказка: `ttest_ind(x1, x2, equal_var=False)`.

**Задача 44 (paired t-test).** Один и тот же читатель до и после «напоминания» о возврате. Сравните средние через `ttest_rel`. Ожидаемый вывод: `t = 3.2`, `p = 0.003` (напоминание работает). В комментариях поясните: чем paired отличается от independent (зависимые наблюдения); почему paired мощнее (убирает межиндивидуальную вариабельность); когда нельзя использовать paired (разные люди); как формулируется H₀ (`μ_diff = 0`). Подсказка: `from scipy.stats import ttest_rel; ttest_rel(before, after)`.

**Задача 45 (one-way ANOVA).** Сравните средний срок выдачи по **трём** жанрам. Выполните `f_oneway` и интерпретируйте. Ожидаемый вывод: `F = 3.8`, `p = 0.024` (есть разница хотя бы между двумя). В комментариях поясните: почему нельзя делать 3 `t-test` (инфляция α, задача 39); что такое `F`-статистика (отношение межгрупповой и внутригрупповой дисперсии); почему ANOVA не говорит, **какие** группы различаются (нужен `post-hoc`, задача 46); предпосылки ANOVA (нормальность, гомогенность дисперсий). Подсказка: `from scipy.stats import f_oneway; f_oneway(g1, g2, g3)`.

**Задача 46 (post-hoc: Tukey HSD).** После значимой ANOVA выясните, какие именно пары различаются. Выполните `pairwise_tukeyhsd`. Ожидаемый вывод: таблица с `p-adj` по каждой паре; значимая — только `роман vs фантастика`. В комментариях поясните: зачем post-hoc (ANOVA — «есть ли хоть какая-то разница»); почему `Tukey HSD` корректирует α (family-wise); чем `Tukey` отличается от `Bonferroni` (менее консервативен); как читать `p-adj` и `reject`. Подсказка: `from statsmodels.stats.multicomp import pairwise_tukeyhsd; pairwise_tukeyhsd(df["days_held"], df["genre"])`.

**Задача 47 (проверка предпосылок).** Перед `t-test`/ANOVA проверьте: нормальность (Shapiro, QQ-plot) и гомогенность дисперсий (Levene). Ожидаемый вывод: `Shapiro p = 0.03` (не нормально), `Levene p = 0.21` (дисперсии равны). В комментариях поясните: почему предпосылки важны; что делать при нарушении нормальности (непараметрические тесты, задача 49; или `n > 30` по ЦПТ); чем `Levene` лучше `Bartlett` (устойчив к ненормальности); что делать при гетерогенности (Welch ANOVA, `Games-Howell`). Подсказка: `from scipy.stats import shapiro, levene`.

**Задача 48 (statsmodels: t-test с ковариатами).** Выполните `t-test` через `statsmodels` с формулой (`days_held ~ group`). Ожидаемый вывод: таблица `coef`, `std err`, `t`, `P>|t|`, `CI`. В комментариях поясните: зачем `statsmodels` вместо `scipy` (ковариаты, категориальные признаки, отчёт); что такое `OLS` и как он связан с `t-test`; как читать `coef` (разница средних); как расширить до ANCOVA (добавить ковариату). Подсказка: `import statsmodels.formula.api as smf; smf.ols("days_held ~ C(group)", data=df).fit().summary()`.

---

## 🔵 Уровень 7. Непараметрические и категориальные (49–56)

**Что изучается:** χ² (независимость, согласие), Mann-Whitney, Wilcoxon, Kruskal-Wallis, Spearman, Kendall, когда непараметрические лучше.

**Задача 49 (χ²: независимость).** Проверьте: зависит ли жанр книги от зала (таблица сопряжённости `genre × hall`)? Выполните `chi2_contingency`. Ожидаемый вывод: `χ² = 12.4`, `p = 0.006` (зависит), `df = 4`. В комментариях поясните: формулу `χ² = Σ (O − E)² / E`; как считать `df = (r−1)(c−1)`; что такое «таблица сопряжённости»; почему χ² не говорит о **направлении** связи (только «есть/нет»); как посчитать `Cramér's V` (сила связи). Подсказка: `from scipy.stats import chi2_contingency; chi2_contingency(pd.crosstab(df["genre"], df["hall"]))`.

**Задача 50 (χ²: согласие).** Проверьте: соответствует ли распределение возвратов гипотезе «30% / 30% / 40%»? Выполните `chisquare`. Ожидаемый вывод: `χ² = 5.2`, `p = 0.07` (не отвергаем H₀). В комментариях поясните: чем тест на согласие отличается от теста на независимость; как задать ожидаемые частоты; почему ожидаемые частоты должны быть ≥ 5 (иначе нужен `Fisher exact`, задача 51). Подсказка: `from scipy.stats import chisquare; chisquare(f_obs=[30, 35, 35], f_exp=[30, 30, 40])`.

**Задача 51 (Fisher exact).** В маленькой выборке (2×2) проверьте независимость через `fisher_exact`. Ожидаемый вывод: `odds_ratio = 2.5`, `p = 0.04`. В комментариях поясните: когда χ² неприменим (маленькие ожидаемые частоты); чем `Fisher` отличается (`точный` тест, не асимптотический); что такое `odds ratio`; почему `Fisher` дорогой (`n!`). Подсказка: `from scipy.stats import fisher_exact; fisher_exact([[10, 5], [3, 12]])`.

**Задача 52 (Mann-Whitney U).** Сравните сроки выдачи в двух залах, но распределение ненормальное. Выполните `mannwhitneyu`. Ожидаемый вывод: `U = 1234`, `p = 0.03`. В комментариях поясните: что проверяет Mann-Whitney (сдвиг распределений, не только средних); чем отличается от `t-test` (непараметрический, устойчив к выбросам); когда `t-test` всё равно лучше (нормальность, `n` большое); как интерпретировать `U`. Подсказка: `from scipy.stats import mannwhitneyu; mannwhitneyu(x1, x2, alternative="two-sided")`.

**Задача 53 (Wilcoxon signed-rank).** Парное сравнение «до/после» при ненормальных данных. Выполните `wilcoxon`. Ожидаемый вывод: `W = 120`, `p = 0.01`. В комментариях поясните: чем отличается от paired t-test (непараметрический аналог); когда `Wilcoxon` лучше (выбросы, малые выборки); когда `paired t-test` всё равно ок (`n > 30`, нормальность); как интерпретировать `W`. Подсказка: `from scipy.stats import wilcoxon; wilcoxon(before, after)`.

**Задача 54 (Kruskal-Wallis).** Сравните три жанра, распределение ненормальное. Выполните `kruskal`. Ожидаемый вывод: `H = 8.4`, `p = 0.015`. В комментариях поясните: непараметрический аналог ANOVA; что такое `H` (ранговая статистика); когда `Kruskal` лучше ANOVA; post-hoc для `Kruskal` (`Dunn's test`, `pairwise Mann-Whitney` с поправкой). Подсказка: `from scipy.stats import kruskal; kruskal(g1, g2, g3)`.

**Задача 55 (Spearman).** Проверьте корреляцию `days_held` и `fine` — монотонную, но не обязательно линейную. Выполните `spearmanr`. Ожидаемый вывод: `ρ = 0.62`, `p = 0.001`. В комментариях поясните: чем `Spearman` отличается от `Pearson` (ранговая корреляция, устойчива к выбросам и нелинейности); когда `Spearman` лучше (ординальные данные, выбросы); почему `ρ` — это «корреляция Пирсона на рангах»; что значит «монотонная, но нелинейная». Подсказка: `from scipy.stats import spearmanr; spearmanr(x, y)`.

**Задача 56 (Kendall).** Сравните `Kendall τ` с `Spearman ρ` на тех же данных. Ожидаемый вывод: `τ = 0.45` vs `ρ = 0.62`. В комментариях поясните: чем `Kendall` отличается (`concordant/discordant` пары, лучше для малых выборок); почему `τ` обычно меньше `ρ`; когда `Kendall` предпочтительнее (маленькие `n`, много «связок» — ties); как интерпретировать. Подсказка: `from scipy.stats import kendalltau; kendalltau(x, y)`.

---

## 🟠 Уровень 8. Корреляция и регрессия (57–64)

**Что изучается:** Pearson, Spearman, Kendall, корреляция ≠ причинность, линейная регрессия (`scipy.stats.linregress`), R², остатки, `statsmodels` (OLS), интерпретация коэффициентов, мультиколлинеарность.

**Задача 57 (Pearson).** Проверьте линейную корреляцию `days_held` и `fine`. Выполните `pearsonr`. Ожидаемый вывод: `r = 0.58`, `p < 0.001`. В комментариях поясните: формулу `r = cov(x,y) / (σx·σy)`; диапазон `[−1, 1]`; интерпретацию (0.1 слабая, 0.3 средняя, 0.5 сильная); почему `r` чувствителен к выбросам; чем `r` отличается от `R²` регрессии (`R² = r²` для простой линейной). Подсказка: `from scipy.stats import pearsonr; pearsonr(x, y)`.

**Задача 58 (корреляция ≠ причинность).** Приведите 3 примера ложной корреляции (или сгенерируйте `x = random`, `y = 2*x + noise`, но `y` и третья переменная `z` коррелируют). Ожидаемый вывод: текстовое объяснение + пример с confounding variable. В комментариях поясните: почему «корреляция ≠ причинность»; что такое `confounder` (скрытая третья переменная); примеры (продажи мороженого и утопления — общий фактор «лето»); как установить причинность (`RCT` — рандомизированный контролируемый эксперимент, `instrumental variables`, `diff-in-diff`); почему в A/B (задача 65) можно говорить о причинности (рандомизация). Подсказка: сгенерируйте `z ~ N(0,1); x = z + noise; y = z + noise` — `x` и `y` коррелируют, но причинной связи нет.

**Задача 59 (простая линейная регрессия).** Постройте модель `fine ~ days_held` через `linregress`. Выведите `slope`, `intercept`, `R²`, `p-value`. Ожидаемый вывод: `slope = 48.2`, `intercept = 12.5`, `R² = 0.34`, `p < 0.001`. В комментариях поясните: что такое `slope` (на сколько `fine` растёт при +1 день); что такое `intercept` (значение `fine` при `days_held = 0` — часто бессмысленно); что такое `R²` (доля дисперсии `y`, объяснённая моделью); почему `p-value` для `slope` — это тест `H₀: slope = 0`. Подсказка: `from scipy.stats import linregress; linregress(x, y)`.

**Задача 60 (остатки).** Проверьте предпосылки регрессии: постройте остатки против предсказанных значений и QQ-plot остатков. Ожидаемый вывод: остатки без паттерна, QQ-plot близок к прямой. В комментариях поясните: зачем анализировать остатки (проверка линейности, гомоскедастичности, нормальности); что такое `heteroscedasticity` (разный разброс остатков) и почему это проблема; как выглядит «плохой» остаточный график (воронка, кривая); что делать при нарушении (лог-преобразование, нелинейная модель). Подсказка: `resid = y - (slope * x + intercept); plt.scatter(y_pred, resid)`.

**Задача 61 (statsmodels OLS: несколько предикторов).** Постройте множественную регрессию `fine ~ days_held + days_overdue + C(genre)` через `statsmodels`. Выведите `coef`, `std err`, `t`, `P>|t|`, `R²`, `Adj. R²`. Ожидаемый вывод: таблица с коэффициентами, `R² = 0.42`. В комментариях поясните: зачем `statsmodels` (интерпретация, отчёты); что такое `coef` (при условии «все остальные фиксированы»); почему `Adj. R²` меньше `R²` (штраф за число предикторов); как интерпретировать `C(genre)` (dummy-переменные, baseline-категория). Подсказка: `import statsmodels.formula.api as smf; smf.ols("fine ~ days_held + days_overdue + C(genre)", data=df).fit().summary()`.

**Задача 62 (мультиколлинеарность).** Проверьте корреляцию между предикторами через `VIF`. Ожидаемый вывод: `VIF(days_held) = 1.2`, `VIF(days_overdue) = 4.8`, `VIF(fine_lag) = 12.3` (высокая). В комментариях поясните: что такое мультиколлинеарность (предикторы коррелируют между собой); почему это проблема (нестабильные коэффициенты, «раздутые» `se`); правило `VIF > 10` — тревога; что делать (удалить один из предикторов, PCA, ridge-регуляризация). Подсказка: `from statsmodels.stats.outliers_influence import variance_inflation_factor; variance_inflation_factor(X.values, i)`.

**Задача 63 (логистическая регрессия).** Постройте модель `returned_on_time ~ days_held + genre` (бинарный исход). Выведите `coef`, `odds ratio`, `p-value`, `AUC`. Ожидаемый вывод: `OR(days_held) = 0.85`, `p < 0.001`, `AUC = 0.72`. В комментариях поясните: чем логистическая регрессия отличается от линейной (логит-преобразование, `p ∈ [0, 1]`); что такое `odds ratio` (`OR > 1` — увеличивает шанс, `OR < 1` — уменьшает); почему `coef` не интерпретируется напрямую как «изменение вероятности»; что такое `AUC` (0.5 — случайно, 1 — идеально). Подсказка: `import statsmodels.formula.api as smf; smf.logit("returned_on_time ~ days_held + C(genre)", data=df).fit()`.

**Задача 64 (R²: интерпретация).** Сравните две модели: `R² = 0.85` vs `R² = 0.30`. Что это значит? Ожидаемый вывод: текстовое объяснение + пример, где `R² = 0.85` — переобучение (или тривиальная задача), а `R² = 0.30` — норм для социальных наук. В комментариях поясните: `R²` — доля объяснённой дисперсии; почему высокий `R²` не всегда хорошо (переобучение, data leakage); почему в реальных задачах `R² = 0.3` — часто отлично; что такое `Adjusted R²` и когда он важнее; почему `R²` нельзя сравнивать между разными датасетами. Подсказка: `model.rsquared`, `model.rsquared_adj`.

---

## 🔴 Уровень 9. A/B-тесты (65–72)

**Что изучается:** MDE, размер выборки, мощность, peeking problem, множественные сравнения (`Bonferroni`, `FDR`), sequential tests, байесовский A/B, симуляция A/B.

**Задача 65 (A/B: постановка).** Библиотека тестирует две рассылки (A — стандартная, B — с напоминанием). Метрика — доля возвратов вовремя. Сформулируйте: H₀, H₁, метрику, MDE, α, мощность. Ожидаемый вывод: `H₀: p_A = p_B`, `H₁: p_A ≠ p_B`, `MDE = 2 pp`, `α = 0.05`, `power = 0.8`. В комментариях поясните: что такое MDE (`minimum detectable effect`); почему MDE задаётся **до** эксперимента; как связаны MDE, `n`, `α`, `power`; почему нельзя «посмотреть данные и решить» (peeking, задача 67). Подсказка: MDE = минимальный эффект, который тест способен обнаружить.

**Задача 66 (размер выборки для A/B).** Вычислите `n` на группу для A/B с `p_baseline = 0.30`, `MDE = 0.02` (2 pp), `α = 0.05`, `power = 0.8`. Ожидаемый вывод: `n ≈ 6300` на группу. В комментариях поясните: формулу для proportions (`n = (z_{α/2} + z_β)² · (p₁(1−p₁) + p₂(1−p₂)) / (p₂ − p₁)²`); почему размер растёт как `1/MDE²` (уменьшить MDE в 2 раза → `n` в 4 раза); как `statsmodels` упрощает (`NormalIndPower`, `proportion_effectsize`); почему маленький MDE требует огромных выборок. Подсказка: `from statsmodels.stats.power import NormalIndPower; from statsmodels.stats.proportion import proportion_effectsize; effect = proportion_effectsize(0.32, 0.30); NormalIndPower().solve_power(effect, power=0.8, alpha=0.05, ratio=1)`.

**Задача 67 (peeking problem).** Продемонстрируйте симуляцией: если проверять `p-value` каждые 100 наблюдений (10 раз подряд), FWER растёт с 5% до ~20%. Ожидаемый вывод: `FWER ≈ 0.19` (а не 0.05). В комментариях поясните: что такое peeking (подглядывание); почему «остановиться, когда p < 0.05» — p-hacking; как исправить (`sequential tests` — задача 70, `alpha-spending`, `group sequential design`); почему в индустрии A/B платформы (Optimizely, VWO) используют sequential. Подсказка: `sum(any(ttest_ind(np.random.normal(0,1,100), np.random.normal(0,1,100)).pvalue < 0.05 for _ in range(10)) for _ in range(10000)) / 10000`.

**Задача 68 (A/B: t-test для proportions).** Проведите `t-test` (или `z-test`) для двух долей. Выполните `proportions_ztest`. Ожидаемый вывод: `z = 2.1`, `p = 0.036` (значимо). В комментариях поясните: чем `z-test` для пропорций отличается от `t-test` для средних; почему при больших `n` `t → z`; как `statsmodels` считает `z` и `p`; почему для маленьких `n` нужен `Fisher` (задача 51). Подсказка: `from statsmodels.stats.proportion import proportions_ztest; proportions_ztest([conv_a, conv_b], [n_a, n_b])`.

**Задача 69 (A/B: несколько метрик).** Вы тестируете 5 метрик (CTR, CR, retention, ARPU, время сессии). После A/B 3 из 5 «значимы». Как интерпретировать? Ожидаемый вывод: текстовый разбор + поправка `Bonferroni` (`α/5 = 0.01`), после которой значимых меньше. В комментариях поясните: почему множественные сравнения инфлируют FWER (задача 39); чем `Bonferroni` (консервативен) отличается от `Holm` (лучше); что такое `FDR` (false discovery rate) и когда он уместен (exploratory, много метрик); почему «3 из 5 значимы» без поправки — подозрительно. Подсказка: `from statsmodels.stats.multitest import multipletests; multipletests(p_values, method="bonferroni")`.

**Задача 70 (sequential test).** Реализуйте простой sequential test: проверяйте `p-value` после каждой партии, но с границей `α_t = α · t/T` (alpha-spending). Продемонстрируйте, что FWER ≤ 0.05. Ожидаемый вывод: `FWER ≈ 0.05` (контролируется). В комментариях поясните: идею `alpha-spending` (Pocock, O'Brien-Fleming); почему sequential позволяет остановиться раньше без инфляции α; чем `SPRT` (Wald) отличается; когда sequential незаменим (A/B в индустрии, клинические испытания). Подсказка: используйте пакет `sequential` или реализуйте вручную с `alpha_t`.

**Задача 71 (байесовский A/B).** Реализуйте байесовский A/B: `Beta(1, 1)` prior, обновление по данным, вероятность `P(p_B > p_A)`. Ожидаемый вывод: `P(p_B > p_A) = 0.92` — «B скорее лучше». В комментариях поясните: чем байесовский A/B отличается от частотного (не `p-value`, а вероятность превосходства); зачем `Beta` prior (сопряжён с биномиальным); как выбрать prior (`Beta(1,1)` — неинформативный, `Beta(a,b)` — исторические данные); почему байесовский A/B удобнее для бизнеса («вероятность, что B лучше — 92%»); какие ограничения (выбор prior, slower для простых решений). Подсказка: `import pymc as pm` или вручную: `samples_a = np.random.beta(1 + conv_a, 1 + n_a - conv_a, 100000); samples_b = ...; (samples_b > samples_a).mean()`.

**Задача 72 (симуляция A/B end-to-end).** Проведите полную симуляцию A/B: сгенерируйте данные для `p_A = 0.30`, `p_B = 0.32`, `n = 6300` на группу, выполните `z-test`, постройте график `p-value` от `n` при последовательном добавлении данных. Ожидаемый вывод: `p-value` падает ниже 0.05 после ~n ≈ 6000; график показывает «скачки» (из-за peeking). В комментариях поясните: как моделировать конверсии (`np.random.binomial`); почему результат меняется от запуска к запуску (вариабельность); почему нужна мощность 0.8 (в 20% случаев не найдём реальный эффект); как визуализировать `p-value` по ходу эксперимента и почему это опасно (peeking). Подсказка: `a = np.random.binomial(1, 0.30, 6300); b = np.random.binomial(1, 0.32, 6300); proportions_ztest([a.sum(), b.sum()], [6300, 6300])`.

---

## ⚫ Уровень 10. Продвинутое (73–80)

**Что изучается:** bootstrap-тесты, permutation tests, `statsmodels` для сложных моделей, `p-value` pitfalls, `effect size` (Cohen's d, η², Cramér's V), `power analysis`, мини-проект полного A/B-анализа.

**Задача 73 (permutation test).** Проверьте разницу средних без предположений о распределении: случайно перемешайте метки групп 10 000 раз, посчитайте распределение разности средних. Ожидаемый вывод: `p_perm = 0.032`. В комментариях поясните: идею permutation test (перестановки под H₀); почему он непараметрический и «точный»; когда лучше `t-test` (быстрее) и когда permutation (малые выборки, негауссовы данные); как связан с bootstrap (permutation — про H₀, bootstrap — про ДИ). Подсказка: `diffs = [np.mean(np.random.permutation(np.concatenate([x, y]))[:len(x)]) - np.mean(...) for _ in range(10000)]; p = (np.abs(diffs) >= abs(obs_diff)).mean()`.

**Задача 74 (bootstrap-тест).** Сравните две группы через bootstrap: постройте 95% ДИ для разности средних. Ожидаемый вывод: `CI = [0.2, 2.8]` (не содержит 0 — значимо). В комментариях поясните: чем bootstrap-тест отличается от permutation (ресемплинг **внутри** групп vs перемешивание); почему bootstrap-ДИ для разности средних асимметричен; когда bootstrap незаменим (медиана, IQR, сложные метрики); почему нужно ≥ 10 000 итераций. Подсказка: `boot_diffs = [np.random.choice(x, len(x), replace=True).mean() - np.random.choice(y, len(y), replace=True).mean() for _ in range(10000)]; np.percentile(boot_diffs, [2.5, 97.5])`.

**Задача 75 (effect size: Cohen's d, η², Cramér's V).** Для трёх сценариев вычислите: (а) `t-test` → `Cohen's d`; (б) ANOVA → `η²`; (в) χ² → `Cramér's V`. Ожидаемый вывод: `d = 0.45` (средний), `η² = 0.06` (средний), `V = 0.21` (слабый-средний). В комментариях поясните: почему `p-value` без effect size бесполезен; правила Cohen для каждого; как `η²` связан с `R²`; почему `Cramér's V` нормирован `[0, 1]`; как включать effect size в отчёты. Подсказка: `d = (x1.mean() - x2.mean()) / pooled_std`; `η² = SS_between / SS_total`; `V = sqrt(χ² / (n * min(r-1, c-1)))`.

**Задача 76 (p-value pitfalls).** Разберите 5 типичных ошибок: (1) `p = 0.05` — «на грани»; (2) `p = 0.06` — «почти значимо»; (3) `p = 0.001` — «эффект огромный»; (4) `p = 0.04` при `n = 10`; (5) `p-value` из одного теста vs из 100. Ожидаемый вывод: текстовый разбор каждой с правильной интерпретацией. В комментариях поясните: почему `p = 0.05` — конвенция, а не магия; почему «почти значимо» — ошибка; почему маленький `p` ≠ большой эффект (зависит от `n`); почему маленькая выборка даёт нестабильный `p`; почему множественные тесты инфлируют α. Подсказка: для каждой ошибки приведите правильную формулировку.

**Задача 77 (power analysis до эксперимента).** Спланируйте A/B: при `p_baseline = 0.30`, `MDE = 0.02`, `α = 0.05`, `power = 0.8` — сколько нужно `n`? А если `MDE = 0.01`? Ожидаемый вывод: `n = 6300` и `n = 25 000`. В комментариях поясните: зачем power analysis **до** эксперимента (не после!); как `statsmodels` считает `n` (`solve_power`); почему «post-hoc power» бессмыслен (зависит от наблюдаемого `p`); как `n` растёт как `1/MDE²`; что делать, если `n` недостижим (увеличить MDE, снизить power, использовать более чувствительную метрику). Подсказка: `NormalIndPower().solve_power(effect_size=proportion_effectsize(0.32, 0.30), power=0.8, alpha=0.05)`.

**Задача 78 (мини-проект: полный A/B-анализ).** Проведите end-to-end A/B-анализ: (1) постановка H₀/H₁, MDE, α, power; (2) power analysis и размер выборки; (3) сбор/симуляция данных; (4) проверка предпосылок; (5) `z-test` + ДИ; (6) effect size (`Cohen's h` для proportions); (7) `Bonferroni`, если несколько метрик; (8) визуализация (`barplot` с ошибками); (9) интерпретация и рекомендация. Ожидаемый вывод: полный отчёт из 5–7 блоков. В комментариях поясните: почему нельзя останавливаться на «p < 0.05»; почему ДИ информативнее `p-value`; как effect size влияет на бизнес-решение; почему визуализация обязательна (для стейкхолдеров); какие ошибки чаще всего делают в A/B (peeking, множественные метрики, недостаточная мощность). Подсказка: структурируйте как `notebook`: Setup → Power → Data → Test → Effect → Interpretation.

**Задача 79 (мета-анализ: intro).** Объедините результаты 5 исследований (mean, se, n) через fixed-effects и random-effects. Ожидаемый вывод: `pooled_effect = 0.42`, `95% CI = [0.30, 0.54]`, `I² = 35%` (умеренная гетерогенность). В комментариях поясните: зачем мета-анализ (объединить доказательства); чем fixed-effects отличается от random-effects; что такое `I²` (доля гетерогенности); как построить forest plot; почему «публикационное смещение» — проблема (funnel plot). Подсказка: `pip install statsmodels` → `statsmodels.stats.meta_analysis`; или `meta-analysis` вручную.

**Задача 80 (финальный проект: статистический отчёт по библиотеке).** Соберите всё: (1) EDA (describe, гистограммы, boxplot); (2) проверка нормальности (Shapiro, QQ); (3) сравнение групп (t-test/ANOVA + effect size); (4) корреляция (Pearson/Spearman + матрица); (5) регрессия (`fine ~ days_held + genre`) с диагностикой; (6) A/B-тест рассылки с power analysis и sequential; (7) визуализации через seaborn; (8) отчёт `report.md` с выводами. Ожидаемый результат: полный аналитический отчёт «под ключ». В комментариях опишите: как разбить на модули (`eda.py`, `tests.py`, `regression.py`, `ab.py`, `report.md`); почему тесты данных важны; как воспроизвести (`seed`); что дальше (dashboard, ML). Подсказка: начните с EDA, добавляйте по одному блоку; каждый блок — с выводами «что это значит для бизнеса».

---
